In [6]:
import numpy as np
import matplotlib . pyplot as plt
# --- Physical parameters ( Table 1) ---
Lx , Ly = 600 *1e3 , 240* 1e3 # domain size [ m ]
Nx , Ny = 256 , 102 # grid resolution
g_prime = 0.3 # reduced gravity [ m / s ^2]
H0 = 1000.0 # mean layer depth [ m ]
U0 = 8.0 # inflow wind speed [ m / s ]
R0 = 20 *1e3 # island radius [ m ]
Cd = 0.02 # drag coefficient [1/ s ]
nu = 1000.0 # eddy viscosity [ m ^2/ s ]
dt = 30.0 # time step [ s ]
# --- Grid : x has walls ( keep endpoints ) , y is periodic ( drop last point ) ---
x = np . linspace (0 , Lx , Nx )
y = np . linspace (0 , Ly , Ny , endpoint = False )
dx = x [1] - x [0]
dy = y [1] - y [0]
X , Y = np . meshgrid (x , y ) # shape ( Ny , Nx ) : x along columns , y along rows
# --- Initial fields ---
rng = np.random.default_rng(0)
h = np.full (( Ny , Nx ) , H0 )
u = np.full (( Ny , Nx ) , U0 ) + 1*1e-3* U0 * rng.standard_normal (( Ny , Nx ) )
v = np.zeros (( Ny , Nx ) ) + 1*1e-3* U0 * rng.standard_normal (( Ny , Nx ) )
# --- Gaussian mountain mask C (x , y ) , centred at ( Lx /3 , Ly /2) ---
xc , yc = Lx /3 , Ly /2
C = np.exp ( -((( X - xc ) **2 + ( Y - yc ) **2) / (0.35* R0 ) **2) )



In [7]:
def ddx ( q ) :
    out = np . zeros_like ( q )
    out [: , 1: -1] = ( q [: , 2:] - q [: , : -2]) / (2* dx ) # central , O (dx ^2)
    out [: , 0] = ( q [: , 1] - q [: , 0]) / dx # forward at x = 0
    out [: , -1] = ( q [: , -1] - q [: , -2]) / dx # backward at x = Lx
    return out
def ddy ( q ) :
# y is periodic -> centred difference with np . roll along the rows
    return ( np . roll (q , -1 , axis =0) - np . roll (q , 1 , axis =0) ) / (2* dy )
def laplacian ( q ) :
    d2x = np . zeros_like ( q )
    d2x [: , 1: -1] = ( q [: , 2:] - 2* q [: , 1: -1] + q [: , : -2]) / dx **2
    d2x [: , 0] = ( q [: , 0] - 2* q [: , 1] + q [: , 2]) / dx **2 # one - sided wall
    d2x [: , -1] = ( q [: , -1] - 2* q [: , -2] + q [: , -3]) / dx **2 # one - sided wall
    d2y = ( np . roll (q , -1 , axis =0) - 2* q + np . roll (q , 1 , axis =0) ) /dy **2
    return d2x + d2y

In [5]:
q = np . sin (2* np . pi * X / Lx )
num = ddx ( q )
ana = (2* np . pi / Lx ) * np . cos (2* np . pi * X / Lx )
print ( " max relative error : " , np . abs ( num - ana ) . max () / (2* np . pi / Lx ) )

 max relative error :  0.00010118472178036849


In [ ]:
def rhs(h,u,v):
    
    dh_dt = -ddx(h*u) - ddy(h*v)

    du_dt = (-u * ddx(u)
               - v * ddy(u)
               - g_prime * ddx(h)
               + nu * laplacian(u)
               - Cd * C * u)
    
    dv_dt = (-u * ddx(v)
               - v * ddy(v)
               - g_prime * ddy(h)
               + nu * laplacian(v)
               - Cd * C * v)
    
    return dh_dt, du_dt, dv_dt

    

In [9]:
def rk2(x,h,rhs,*y):
    k1 = rhs (x * y)
    k2 = rhs(x + h/2, y + h/2 * k1 )
    y_new = y + h*k2

    return y_new
    